# Experiment 049: GPT-2 XL MC-SFT symmetry (Colab)

This runner trains the fixed p2, p8, p32 and warm p8 families. Run seed 0 first, then set the seed tuple to 1 and 2 in a new session.


## 0. Compute units and Drive


In [1]:
NOTEBOOK_NAME = '049_colab_gpt2_xl_mc_sft'
import glob, json, os, time
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

session_id = globals().get('session_id', time.strftime('%Y%m%d_%H%M%S'))
cu_before = 385
unit_rate = 5.3
print('session', session_id, '| cu_before =', cu_before if cu_before is not None else 'FILL ME')

from google.colab import drive
drive.mount('/content/drive')
ROOT = '/content/drive/MyDrive/inm363'
RESULTS = os.path.join(ROOT, 'results')
OPEN_UNITS = os.path.join(RESULTS, 'open_cu_sessions')
UNITS_LOG = os.path.join(RESULTS, 'colab_units.jsonl')
os.makedirs(OPEN_UNITS, exist_ok=True)
old_sessions = [path for path in glob.glob(os.path.join(OPEN_UNITS, '*.json'))
                if not path.endswith(f'{session_id}.json')]
open_path = os.path.join(OPEN_UNITS, f'{session_id}.json')
if cu_before is None or unit_rate is None:
    print('fill cu_before and unit_rate, then rerun this cell')
else:
    for old_path in old_sessions:
        with open(old_path) as handle:
            old = json.load(handle)
        old['ts_end'] = time.strftime('%Y-%m-%d %H:%M')
        old['colab_cu_after'] = cu_before
        old['colab_cu_used'] = round(old['colab_cu_before'] - cu_before, 2)
        old['closed_on_resume'] = True
        with open(UNITS_LOG, 'a') as handle:
            handle.write(json.dumps(old) + '\n')
        os.remove(old_path)
    if not os.path.exists(open_path):
        with open(open_path, 'w') as handle:
            json.dump({'session_id': session_id, 'ts_start': time.strftime('%Y-%m-%d %H:%M'),
                       'notebook': NOTEBOOK_NAME, 'colab_cu_before': cu_before,
                       'unit_rate': unit_rate}, handle, indent=2)


session 20260831_173551 | cu_before = 385
Mounted at /content/drive


## 1. Repository and dependencies


In [2]:
import base64
import getpass
import os
from pathlib import Path
import subprocess
import sys

REPO_SLUG = 'hastov-lab/rl-grpo-or-sft-for-llms'
repo_url = f'https://github.com/{REPO_SLUG}.git'
repo_root = Path('/content') / REPO_SLUG.split('/')[-1]


def repo_git(*args):
    git = ['git']
    env = None
    try:
        from google.colab import userdata
    except ImportError:
        userdata = None

    # Private forks can use a read token from Colab Secrets.
    if userdata is not None and REPO_SLUG != 'hastov-lab/rl-grpo-or-sft-for-llms':
        try:
            token = userdata.get('GH_TOKEN')
        except Exception:
            token = None
        if isinstance(token, dict):
            token = token.get('value')
        if not isinstance(token, str) or not token.strip():
            token = getpass.getpass('GitHub token: ')
        if not token.strip():
            raise RuntimeError('This private fork requires a GitHub token with read access.')

        # Keep the token out of the command line and the saved remote URL.
        auth = base64.b64encode(f'x-access-token:{token.strip()}'.encode()).decode()
        env = dict(os.environ, REPO_GIT_AUTH=f'Authorization: Basic {auth}')
        git.append('--config-env=http.https://github.com/.extraHeader=REPO_GIT_AUTH')

    subprocess.run([*git, *args], env=env, check=True)


if repo_root.exists():
    if not (repo_root / '.git').exists():
        raise RuntimeError(f'Expected a Git checkout at {repo_root}.')
    origin = subprocess.check_output(
        ['git', '-C', str(repo_root), 'config', '--local', '--get', 'remote.origin.url'], text=True
    ).strip()
    if origin != repo_url:
        raise RuntimeError(f'The checkout at {repo_root} belongs to a different repository.')
    branch = subprocess.check_output(
        ['git', '-C', str(repo_root), 'branch', '--show-current'], text=True
    ).strip()
    if branch != 'main':
        raise RuntimeError(f'Expected the main branch at {repo_root}.')
    repo_git('-C', str(repo_root), 'pull', '--ff-only', 'origin', 'main')
else:
    repo_git('clone', '--branch', 'main', '--single-branch', '--depth', '1', repo_url, str(repo_root))

# Use the scripts from this checkout even if another experiment is open.
code_root = repo_root / '001_Code'
if not (code_root / 'scripts' / 'run.py').is_file():
    raise RuntimeError(f'Experiment scripts are missing from {code_root}.')
os.chdir(code_root)
print('working in:', os.getcwd())


working in: /content/INM363-IndividualProject-Dissertation/001_Code


In [3]:
import gc, os, subprocess, sys, torch
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'trl==1.7.1', 'transformers==5.5.4', 'peft==0.19.1',
                'accelerate==1.14.0', 'datasets==4.8.4', 'wandb', 'pyyaml'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-q', '-y', 'torchao'], check=True)
scratch_package = os.path.join(repo_root, '002_GPT2_From_Scratch')
if not os.path.isdir(scratch_package):
    scratch_package = os.path.join(repo_root, '003_My GPT From Scratch')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', scratch_package], check=True)
sys.path.insert(0, scratch_package)

from transformers import GPT2LMHeadModel, GPT2TokenizerFast
from gpt2_scratch.weights import reference_checkpoint

reference_name, reference_revision = reference_checkpoint('gpt2-xl')
GPT2TokenizerFast.from_pretrained(reference_name, revision=reference_revision)
reference = GPT2LMHeadModel.from_pretrained(
    reference_name, revision=reference_revision, dtype=torch.float32)
del reference
gc.collect()
print('dependencies installed')


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/689 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 6.43GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/580 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

dependencies installed


In [4]:
import getpass, os
wandb_key = None
try:
    from google.colab import userdata
    wandb_key = userdata.get('WANDB_API_KEY')
except Exception:
    wandb_key = None
if not wandb_key:
    wandb_key = getpass.getpass('W&B API key: ').strip()
assert wandb_key, 'WANDB_API_KEY is required'
os.environ['WANDB_API_KEY'] = wandb_key
wandb_key = None


## 2. Registered setup


In [5]:
import json, os, shutil, subprocess, sys, time, torch
sys.path.insert(0, os.getcwd())

from file_utils import read_json
from scripts.grpo_worker import stream_subprocess
from scripts.s12_eval import IntegrityError
from scripts.s49_gpt2_xl_symmetry import (SEEDS_049, arm_state, arm_tag,
                                          check_card_vram, commit_arm_complete,
                                          commit_eval, commit_launch,
                                          commit_preflight, config_path,
                                          eval_state, registered_configs,
                                          training_record, verify_launch)

assert cu_before is not None and unit_rate is not None, 'fill the compute fields first'
assert os.environ.get('PYTORCH_CUDA_ALLOC_CONF') == 'expandable_segments:True'
cfgs = registered_configs()
gpu = subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True).stdout
total_gib = torch.cuda.get_device_properties(0).total_memory / 1024**3
assert 'A100' in gpu, 'need an A100 runtime'
assert shutil.disk_usage(ROOT).free / 1024**3 >= 45, 'need at least 45 GiB free on Drive'

TECH = os.path.join(RESULTS, '049_technical')
PREFLIGHT = os.path.join(TECH, '049_preflight.json')
LAUNCH = os.path.join(TECH, '049_launch.json')
BASE_RESULT = os.path.join(RESULTS, '046_COLAB_FULL_GPT2_B15_base')
BASE_RECORD = os.path.join(BASE_RESULT, '046_base_manifest.json')
PARITY = os.path.join(RESULTS, '046_technical', '046_xl_parity.json')
TOKEN_AUDIT = os.path.join(RESULTS, '046_technical', '046_token_audit.json')
PARENT_RESULTS = {
    seed: os.path.join(RESULTS, f'046_COLAB_FULL_GPT2_B15_sft_grpo_seed{seed}')
    for seed in SEEDS_049
}

FAMILIES = ('p2', 'p8', 'p32', 'wsft_p8')
check_card_vram(total_gib, 'p8')
#SEEDS_TO_RUN = (0,)
# all other seeds to finish:
SEEDS_TO_RUN = (1, 2)
assert set(SEEDS_TO_RUN) <= set(SEEDS_049)
os.environ['WANDB_PROJECT'] = cfgs[('p8', 0)].log.wandb_project


## 3. Wiring checks


In [6]:
test_env = os.environ.copy()
test_env['CUDA_VISIBLE_DEVICES'] = ''
checks = subprocess.run([
    sys.executable, '-m', 'pytest', '-q',
    'tests/test_049_gpt2_xl_symmetry.py', 'tests/test_protocol_freeze.py',
], capture_output=True, text=True, env=test_env)
print(checks.stdout)
if checks.stderr:
    print(checks.stderr)
checks.check_returncode()


..............................                                           [100%]
30 passed in 2.11s



## 4. Worker helpers


In [7]:
def active_workers():
    rows = subprocess.run(['ps', '-eo', 'pid=,args='], capture_output=True,
                          text=True, check=True).stdout
    return [row.strip() for row in rows.splitlines()
            if 'scripts/s49_worker.py' in row]

def completed_prediction_pass(path):
    if not os.path.isfile(path):
        return False
    count = 0
    try:
        with open(path) as handle:
            for line in handle:
                row = json.loads(line)
                if not {'subject', 'tier', 'gold'} <= set(row):
                    return False
                count += 1
    except (OSError, TypeError, ValueError):
        return False
    return count == 1884


def observed_test_passes(values):
    if '--scratch-dir' not in values:
        return 0
    scratch = values[values.index('--scratch-dir') + 1]
    names = (f'{branch}{suffix}' for branch in ('sft', 'grpo')
             for suffix in ('_preds.jsonl', '_logprob_preds.jsonl'))
    return sum(completed_prediction_pass(os.path.join(scratch, name))
               for name in names)


def archive_failed_eval(values, status, result_dir):
    if ('--scratch-dir' not in values or status.get('completed') is True):
        return None
    scratch = values[values.index('--scratch-dir') + 1]
    if not os.path.isdir(scratch) or not os.listdir(scratch):
        return None
    archive = os.path.join(
        result_dir, f'failed_eval_{session_id}_{time.time_ns()}')
    shutil.copytree(scratch, archive)
    status['eval_scratch_archive'] = archive
    return archive


def run_worker(values, status_path, result_dir, label, test_passes=0):
    if active_workers():
        raise RuntimeError(f'{label}: another experiment 049 worker is active')
    if os.path.exists(status_path):
        old = read_json(status_path)
        if old.get('completed') is True:
            raise IntegrityError(f'{label}: completed worker has no committed result')
        if old.get('error_type') not in (None, 'KeyboardInterrupt'):
            raise RuntimeError(f'{label}: previous worker failure needs review')
        os.remove(status_path)
    started = time.time()
    process = None
    interrupted = None
    try:
        process = stream_subprocess(values)
    except KeyboardInterrupt as exc:
        interrupted = exc
    finally:
        status = read_json(status_path) if os.path.exists(status_path) else {
            'completed': False, 'error_type': 'MissingStatus'}
        status['returncode'] = process.returncode if process is not None else None
        status['label'] = label
        actual_test_passes = observed_test_passes(values) if test_passes else 0
        status['mmlu_test_passes'] = actual_test_passes
        archive_failed_eval(values, status, result_dir)
        with open(os.path.join(result_dir, '049_attempts.jsonl'), 'a') as handle:
            handle.write(json.dumps(status) + '\n')
        with open(os.path.join(RESULTS, 'compute_ledger.jsonl'), 'a') as handle:
            handle.write(json.dumps({
                'run_tag': os.path.basename(result_dir), 'condition': label,
                'timings_sec': {label: round(time.time() - started, 1)},
                'train_steps': status.get('completed_updates'),
                'completed': status.get('completed') is True,
                'gpu': status.get('gpu'), 'mmlu_test_passes': actual_test_passes,
            }) + '\n')
    if interrupted is not None:
        raise interrupted
    if status.get('completed') is True and actual_test_passes != test_passes:
        raise IntegrityError(f'{label}: completed test passes differ')
    return status

def command(task, cfg_path, status_path, result_dir, **kwargs):
    values = [sys.executable, '-u', 'scripts/s49_worker.py', '--task', task,
              '--config', cfg_path, '--status', status_path,
              '--result-dir', result_dir, '--preflight', PREFLIGHT,
              '--launch-record', LAUNCH]
    for key, value in kwargs.items():
        values.extend(['--' + key.replace('_', '-'), value])
    return values

def fresh_scratch(label):
    path = f'/content/{label}_{session_id}'
    shutil.rmtree(path, ignore_errors=True)
    return path


## 5. Source records


In [8]:
os.makedirs(TECH, exist_ok=True)
preflight = commit_preflight(
    PREFLIGHT, BASE_RECORD, BASE_RESULT, PARITY, TOKEN_AUDIT, PARENT_RESULTS)
launch = commit_launch(LAUNCH, PREFLIGHT, cu_before)
verify_launch(LAUNCH, PREFLIGHT)
print('preflight ready:', len(preflight['roster']), 'arms')


preflight ready: 15 arms


## 6. Fixed MC-SFT arms


In [9]:
def run_arm(family, seed):
    cfg = cfgs[(family, seed)]
    cfg_path = config_path(family, seed)
    tag = arm_tag(family, seed)
    result_dir = os.path.join(RESULTS, tag)
    output_dir = os.path.join(ROOT, 'checkpoints', tag, 'sft')
    os.makedirs(result_dir, exist_ok=True)

    state = arm_state(cfg, result_dir, PREFLIGHT)
    if state['state'] == 'committed':
        return 'committed'
    if state['state'] != 'absent':
        raise RuntimeError(f'{tag}: previous state needs review: {state}')

    record_path = os.path.join(result_dir, '049_sft_training.json')
    if not os.path.exists(record_path):
        status_path = os.path.join(result_dir, 'sft_worker_status.json')
        status = run_worker(command(
            'sft', cfg_path, status_path, result_dir, out_dir=output_dir),
            status_path, result_dir, f'{tag}_sft')
        if status.get('completed') is not True:
            raise RuntimeError(f'{tag}: SFT failed: {status}')
    binding = training_record(cfg, result_dir)

    state = eval_state(cfg, result_dir, binding)
    if state == 'tampered':
        raise IntegrityError(f'{tag}: SFT endpoint changed')
    if state == 'absent':
        status_path = os.path.join(result_dir, 'sft_eval_status.json')
        if os.path.exists(status_path) and read_json(status_path).get('completed') is True:
            raise IntegrityError(f'{tag}: completed evaluation has no manifest')
        scratch = fresh_scratch(f'049_{family}_{seed}_eval')
        status = run_worker(command(
            'eval', cfg_path, status_path, result_dir,
            adapter_dir=output_dir, scratch_dir=scratch),
            status_path, result_dir, f'{tag}_eval', test_passes=2)
        if status.get('completed') is True:
            shutil.rmtree(scratch, ignore_errors=True)
        else:
            print('evaluation scratch kept:', scratch)
            raise RuntimeError(f'{tag}: evaluation failed: {status}')
    commit_arm_complete(cfg, result_dir, PREFLIGHT)
    return 'committed'


outcomes = {}
for seed in SEEDS_TO_RUN:
    for family in FAMILIES:
        outcomes[(family, seed)] = run_arm(family, seed)
print(outcomes)


README.md: 100% 53.2k/53.2k [00:00<00:00, 108MB/s]
dataset_infos.json: 100% 138k/138k [00:00<00:00, 209MB/s]

high_school_psychology/test-00000-of-000(…): downloading bytes:   0% 0.00/92.8k [00:00<?, ?B/s]
high_school_psychology/test-00000-of-000(…): downloading bytes: 100% 91.9k/91.9k [00:00<00:00, 224kB/s, 9.15kB/s  ]
high_school_psychology/test-00000-of-000(…): reconstructing file: 100% 92.8k/92.8k [00:00<00:00, 226kB/s, 9.23kB/s  ]

high_school_psychology/validation-00000-(…): downloading bytes:   0% 0.00/15.2k [00:00<?, ?B/s]
high_school_psychology/validation-00000-(…): downloading bytes: 100% 13.4k/13.4k [00:00<00:00, 26.1kB/s, 1.33kB/s  ]
high_school_psychology/validation-00000-(…): reconstructing file: 100% 15.2k/15.2k [00:00<00:00, 29.7kB/s, 1.51kB/s  ]

high_school_psychology/dev-00000-of-0000(…): downloading bytes:   0% 0.00/5.18k [00:00<?, ?B/s]
high_school_psychology/dev-00000-of-0000(…): downloading bytes: 100% 3.21k/3.21k [00:00<00:00, 8.51kB/s,   320B/s  ]
high_school_p

## 7. Close compute units


In [11]:
cu_after = 356.12
print('cu_after =', cu_after if cu_after is not None else 'FILL ME')
if cu_after is None:
    print('fill cu_after, then rerun this cell')
elif os.path.exists(open_path):
    with open(open_path) as handle:
        record = json.load(handle)
    record['ts_end'] = time.strftime('%Y-%m-%d %H:%M')
    record['colab_cu_after'] = cu_after
    record['colab_cu_used'] = round(record['colab_cu_before'] - cu_after, 2)
    record['completed'] = True
    with open(UNITS_LOG, 'a') as handle:
        handle.write(json.dumps(record) + '\n')
    os.remove(open_path)
    print('logged:', record)


cu_after = 356.12
logged: {'session_id': '20260831_173551', 'ts_start': '2026-08-31 17:36', 'notebook': '049_colab_gpt2_xl_mc_sft', 'colab_cu_before': 385, 'unit_rate': 5.3, 'ts_end': '2026-08-31 23:16', 'colab_cu_after': 356.12, 'colab_cu_used': 28.88, 'completed': True}
